In [332]:
import os
import pandas as pd
import numpy as np

In [333]:
df_casernes = pd.read_csv('casernes.csv')
print(df_casernes.shape)
df_casernes.head()

(68, 11)


,CASERNE,NO_CIVIQUE,RUE,LATITUDE,LONGITUDE,ARRONDISSEMENT,VILLE,DATE_DEBUT,DATE_FIN,MTM8_X,MTM8_Y
0,3,256,rue Young,45.493454,-73.560172,LE SUD-OUEST,NaN,2015-01-01T07:00:00,NaN,300097.0,5039283.0
1,15,1255,rue de la Sucrerie,45.484419,-73.560917,LE SUD-OUEST,NaN,2015-01-01T07:00:00,NaN,300038.0,5038279.0
2,23,523,place Saint-Henri,45.477820,-73.585257,LE SUD-OUEST,NaN,2015-01-01T07:00:00,NaN,298134.6,5037547.3
3,33,6040,boulevard Monk,45.457841,-73.595450,LE SUD-OUEST,NaN,2015-01-01T07:00:00,NaN,297335.0,5035328.0
4,9,8100,boulevard Saint-Michel,45.563748,-73.610169,VILLERAY-SAINT-MICHEL-PARC-EXTENSION,NaN,2019-05-06T07:00:00,NaN,296200.0,5047099.0


In [334]:
# Combine the municipalities and boroughs into a single column
df_casernes = df_casernes[['CASERNE', 'ARRONDISSEMENT', 'VILLE']]
df_casernes['MUNICIPALITY/BOROUGH'] = df_casernes['ARRONDISSEMENT'].combine_first(df_casernes['VILLE'])
df_casernes = df_casernes[['CASERNE', 'MUNICIPALITY/BOROUGH']]
df_casernes.loc[:, 'MUNICIPALITY/BOROUGH'] = df_casernes.loc[:, 'MUNICIPALITY/BOROUGH'].str.title()

In [335]:
df_population = pd.read_csv('municipalities-boroughs-population.csv')
df_population.head()

,CODE,MUNICIPALITY/BOROUGH,2005,2006,2007,2008,2009,2010,2011,2012,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,66112,Baie-D'Urfé,3965,3965,3953,3984,3946,3928,3881,3882,...,3873,3900,3847,3890,3907,3922,3944,3962,3889,3710
1,66107,Beaconsfield,20183,20183,20262,19517,19168,19378,19816,20046,...,19847,19801,19957,20075,19588,19977,19942,20124,19755,19290
2,66058,Côte-Saint-Luc,31739,31739,31764,31965,31458,32294,32703,33140,...,33392,33847,34066,34464,33644,34761,35117,34911,35419,37833
3,66142,Dollard-Des-Ormeaux,50738,50738,50707,49940,49430,49445,49902,50154,...,50700,50789,51050,51200,49424,50453,50302,49696,49908,50171
4,66087,Dorval,18276,18311,18297,18445,18238,18231,18615,19013,...,19170,19426,19579,19763,19535,20040,19907,19735,19993,20382


In [336]:
years = df_population.iloc[:, 2:].columns.to_list()

# Sainte Anne and Senneville are part of the same caserne territory so incorporate their populations together
df_sainte_anne = df_population.loc[df_population['MUNICIPALITY/BOROUGH'] == 'Sainte-Anne-De-Bellevue']
df_senneville = df_population.loc[df_population['MUNICIPALITY/BOROUGH'] == 'Senneville']

sum = df_sainte_anne.iloc[0, 2:] + df_senneville.iloc[0, 2:]

for year in years:
  df_population.loc[df_population['MUNICIPALITY/BOROUGH'] == 'Sainte-Anne-De-Bellevue', year] = sum[year]

df_population = df_population[df_population['MUNICIPALITY/BOROUGH'] != 'Senneville']

In [337]:
years = df_population.iloc[:, 2:].columns.to_list()

# Dorval and L'Île-Dorval are part of the same caserne territory so incorporate their populations together
df_dorval = df_population.loc[df_population['MUNICIPALITY/BOROUGH'] == 'Dorval']
df_lile_dorval = df_population.loc[df_population['MUNICIPALITY/BOROUGH'] == "L'Île-Dorval"]

sum = df_dorval.iloc[0, 2:] + df_lile_dorval.iloc[0, 2:]

for year in years:
  df_population.loc[df_population['MUNICIPALITY/BOROUGH'] == 'Dorval', year] = sum[year]

df_population = df_population[df_population['MUNICIPALITY/BOROUGH'] != "L'Île-Dorval"]

In [338]:
df_area = pd.read_csv('municipalities-boroughs-area.csv')
df_area.head()

,CODE,MUNICIPALITY/BOROUGH,AREA_SQUARE_KM
0,66112,Baie-D'Urfé,6.0
1,66107,Beaconsfield,11.0
2,66058,Côte-Saint-Luc,6.9
3,66142,Dollard-Des-Ormeaux,15.2
4,66087,Dorval,20.9


In [339]:
# Sainte Anne and Senneville are part of the same caserne territory so incorporate their area together
df_sainte_anne = df_area.loc[df_area['MUNICIPALITY/BOROUGH'] == 'Sainte-Anne-De-Bellevue']
df_senneville = df_area.loc[df_area['MUNICIPALITY/BOROUGH'] == 'Senneville']

sum = df_sainte_anne.iloc[0, 2:] + df_senneville.iloc[0, 2:]

df_area.loc[df_area['MUNICIPALITY/BOROUGH'] == 'Sainte-Anne-De-Bellevue', 'AREA_SQUARE_KM'] = sum['AREA_SQUARE_KM']

df_area = df_area[df_area['MUNICIPALITY/BOROUGH'] != 'Senneville']

In [340]:
# Dorval and L'Île-Dorval part of the same caserne territory so incorporate their area together
df_dorval = df_area.loc[df_area['MUNICIPALITY/BOROUGH'] == 'Dorval']
df_lile_dorval = df_area.loc[df_area['MUNICIPALITY/BOROUGH'] == "L'Île-Dorval"]

sum = df_dorval.iloc[0, 2:] + df_lile_dorval.iloc[0, 2:]

df_area.loc[df_area['MUNICIPALITY/BOROUGH'] == 'Dorval', 'AREA_SQUARE_KM'] = sum['AREA_SQUARE_KM']

df_area = df_area[df_area['MUNICIPALITY/BOROUGH'] != "L'Île-Dorval"]

In [341]:
df_population_density = df_population.iloc[:, :2]

for year in years:
  df_population_density[year] = (df_population[year] / df_area['AREA_SQUARE_KM']).round(1)
df_population_density.head()

,CODE,MUNICIPALITY/BOROUGH,2005,2006,2007,2008,2009,2010,2011,2012,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,66112,Baie-D'Urfé,660.8,660.8,658.8,664.0,657.7,654.7,646.8,647.0,...,645.5,650.0,641.2,648.3,651.2,653.7,657.3,660.3,648.2,618.3
1,66107,Beaconsfield,1834.8,1834.8,1842.0,1774.3,1742.5,1761.6,1801.5,1822.4,...,1804.3,1800.1,1814.3,1825.0,1780.7,1816.1,1812.9,1829.5,1795.9,1753.6
2,66058,Côte-Saint-Luc,4599.9,4599.9,4603.5,4632.6,4559.1,4680.3,4739.6,4802.9,...,4839.4,4905.4,4937.1,4994.8,4875.9,5037.8,5089.4,5059.6,5133.2,5483.0
3,66142,Dollard-Des-Ormeaux,3338.0,3338.0,3336.0,3285.5,3252.0,3253.0,3283.0,3299.6,...,3335.5,3341.4,3358.6,3368.4,3251.6,3319.3,3309.3,3269.5,3283.4,3300.7
4,66087,Dorval,866.3,867.9,867.3,874.2,864.4,864.0,882.2,901.1,...,908.8,920.9,928.2,936.9,926.1,950.0,943.7,935.5,947.8,966.4


In [342]:
df_caserne_population_density = pd.merge(df_casernes, df_population_density, on='MUNICIPALITY/BOROUGH')
print(df_caserne_population_density.shape)
df_caserne_population_density.head()

(68, 23)


,CASERNE,MUNICIPALITY/BOROUGH,CODE,2005,2006,2007,2008,2009,2010,2011,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,3,Le Sud-Ouest,REM20,4615.5,4550.4,4460.4,4650.6,4437.6,4514.4,4548.5,...,4810.8,4969.9,5077.5,5135.2,5188.9,5369.4,5466.1,5390.4,5499.8,5929.2
1,15,Le Sud-Ouest,REM20,4615.5,4550.4,4460.4,4650.6,4437.6,4514.4,4548.5,...,4810.8,4969.9,5077.5,5135.2,5188.9,5369.4,5466.1,5390.4,5499.8,5929.2
2,23,Le Sud-Ouest,REM20,4615.5,4550.4,4460.4,4650.6,4437.6,4514.4,4548.5,...,4810.8,4969.9,5077.5,5135.2,5188.9,5369.4,5466.1,5390.4,5499.8,5929.2
3,33,Le Sud-Ouest,REM20,4615.5,4550.4,4460.4,4650.6,4437.6,4514.4,4548.5,...,4810.8,4969.9,5077.5,5135.2,5188.9,5369.4,5466.1,5390.4,5499.8,5929.2
4,9,Villeray-Saint-Michel-Parc-Extension,REM25,9013.5,9097.0,8857.1,8853.0,8962.8,8597.9,8717.1,...,8971.3,9034.8,9049.5,9066.1,8847.6,8981.9,9005.2,8715.0,8776.6,9345.7


In [343]:
df_population_density.to_csv('caserne-population-density.csv', index=False)